# ForgeAI - full demo builds on a free Kaggle GPU

Runs both demos and the baseline comparison with the **same small local model** (`qwen2.5:3b` on Ollama). The only difference from a laptop run is the hardware: a free T4 GPU.

**Setup (once)**
1. On your PC: `python scripts/make_kaggle_bundle.py` creates `dist/forgeai_kaggle_bundle.zip`.
2. Kaggle → *Datasets → New dataset*: upload that zip, keep it **Private**.
3. Kaggle → *Code → New notebook → File → Import notebook*: choose this `.ipynb`.
4. Notebook sidebar: **Accelerator = GPU T4**, **Internet = On** (needs a phone-verified account), then *Add Input* → your bundle dataset.

**Run it unattended:** *Save Version → Save & Run All (Commit)*. The run keeps going after you close the browser (12 h session limit). When it finishes, open the version → *Output* and download `forgeai_results.zip`.

In [ ]:
# ---- settings
RUN_DEMO1, RUN_DEMO2, RUN_COMPARE = True, True, True
MAX_ITERS = 3
MAX_SECONDS_PER_BUILD = 14400      # 4 h safety cap per build (a GPU build normally needs well under 1 h)
MODEL, EMBED_MODEL = "qwen2.5:3b", "nomic-embed-text"

In [ ]:
# ---- 1. GPU check
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "NO GPU FOUND - set Accelerator = GPU T4 in the notebook settings (it still works on CPU, just slowly).")

In [ ]:
# ---- 2. copy the bundle into a writable folder and install ForgeAI
import glob, os, pathlib, shutil, subprocess, sys, zipfile

WORK = pathlib.Path("/kaggle/working/forgeai")
found = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
if WORK.exists():
    shutil.rmtree(WORK)
if found:                                        # Kaggle normally unpacks uploaded zips
    shutil.copytree(pathlib.Path(found[0]).parent, WORK)
else:                                            # ...but handle a still-zipped upload too
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "Bundle not found: use 'Add Input' to attach your forgeai bundle dataset."
    zipfile.ZipFile(zips[0]).extractall(WORK)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("installed from", WORK)

In [ ]:
# ---- 3. install Ollama (zstd is needed by the current installer)
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ---- 4. start the Ollama server in the background and pull the models
import requests, time

env = dict(os.environ, OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
ollama_log = open("/kaggle/working/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, env=env)
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("ollama server is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/kaggle/working/ollama.log").read()[-3000:])

for m in (MODEL, EMBED_MODEL):
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "pulled" if r.returncode == 0 else "PULL FAILED:\n" + r.stderr[-1500:])

In [ ]:
# ---- 5. point ForgeAI at the local server and smoke-test it
os.environ.update({
    "FORGE_BASE_URL": "http://127.0.0.1:11434/v1",
    "FORGE_MODEL": MODEL,
    "FORGE_EMBED_MODEL": EMBED_MODEL,
    "FORGE_NUM_CTX": "8192",
    "FORGE_REQUEST_TIMEOUT": "300",
    "PYTHONIOENCODING": "utf-8",
    "COLUMNS": "160",
})


def sh(args):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, args)), flush=True)
    p = subprocess.Popen(list(map(str, args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit code {rc}]\n", flush=True)
    return rc


FORGE = [sys.executable, "-m", "forgeai"]
sh(FORGE + ["doctor"])
sh(["ollama", "ps"])          # PROCESSOR column should say GPU

In [ ]:
# ---- 6. Demo 1: Maritime HR & Shore-Leave (expect tools = [])
if RUN_DEMO1:
    t = time.time()
    sh(FORGE + ["build", "-b", "demos/demo1_hr/brief.txt", "-d", "demos/demo1_hr/docs", "-o", "demos/demo1_hr/run",
                "--answers", "demos/demo1_hr/answers.json", "--auto-answer",
                "--max-iters", MAX_ITERS, "--max-seconds", MAX_SECONDS_PER_BUILD])
    print(f"Demo 1 took {(time.time() - t) / 60:.1f} min")

In [ ]:
# ---- 7. Demo 2: Vessel Spares & Procurement (expect tools = csv_lookup + calculator)
if RUN_DEMO2:
    t = time.time()
    sh(FORGE + ["build", "-b", "demos/demo2_spares/brief.txt", "-d", "demos/demo2_spares/docs", "-o", "demos/demo2_spares/run",
                "--answers", "demos/demo2_spares/answers.json", "--auto-answer",
                "--max-iters", MAX_ITERS, "--max-seconds", MAX_SECONDS_PER_BUILD])
    print(f"Demo 2 took {(time.time() - t) / 60:.1f} min")

In [ ]:
# ---- 8. factory vs hand-crafted baseline on the Demo 1 eval set
if RUN_COMPARE and pathlib.Path("demos/demo1_hr/run/agent_definition.json").exists():
    sh(FORGE + ["compare", "demos/demo1_hr/run/agent_definition.json", "demos/demo1_hr/baseline/agent_definition.json",
                "-e", "demos/demo1_hr/run/eval_set.json", "-o", "demos/demo1_hr/baseline_comparison",
                "--name-a", "factory", "--name-b", "baseline"])

In [ ]:
# ---- 9. sanity checks before you download anything
import json

EXPECTED_TOOLS = {"demos/demo1_hr/run": [], "demos/demo2_spares/run": ["calculator", "csv_lookup"]}
for run, want in EXPECTED_TOOLS.items():
    d = pathlib.Path(run)
    if not (d / "run_status.json").exists():
        print(f"{run}: NOT RUN")
        continue
    status = json.loads((d / "run_status.json").read_text())
    line = f"{run}: status={status['status']}"
    if (d / "scorecard.json").exists():
        final = json.loads((d / "scorecard.json").read_text())["final"]
        tools = sorted(json.loads((d / "agent_definition.json").read_text())["selected_tools"])
        cases = len(json.loads((d / "eval_set.json").read_text())["cases"])
        line += (f"  cases={cases}  pass={final['pass_rate']:.0%}  overall={final['overall']:.3f}"
                 f"  tools={tools} {'OK' if tools == want else 'UNEXPECTED (wanted ' + str(want) + ')'}")
    else:
        line += f"  stage={status.get('stage')}  reason={status.get('reason', '')}"
    print(line)
cmp = pathlib.Path("demos/demo1_hr/baseline_comparison/comparison_report.md")
print("\n" + (cmp.read_text()[:1500] if cmp.exists() else "comparison not run"))

In [ ]:
# ---- 10. package the results (unzip at the repo root on your PC)
import zipfile

dirs = ["demos/demo1_hr/run", "demos/demo2_spares/run", "demos/demo1_hr/baseline_comparison"]
res = pathlib.Path("/kaggle/working/forgeai_results.zip")
with zipfile.ZipFile(res, "w", zipfile.ZIP_DEFLATED) as z:
    for d in dirs:
        for f in sorted(pathlib.Path(d).rglob("*")):
            if f.is_file():
                z.write(f, f.as_posix())
# the LLM response cache, so later commands on your PC replay these calls instantly (optional download)
cache = pathlib.Path("/kaggle/working/forgeai_llm_cache.zip")
with zipfile.ZipFile(cache, "w", zipfile.ZIP_DEFLATED) as z:
    for f in pathlib.Path(".forge_cache").rglob("*"):
        if f.is_file():
            z.write(f, f.as_posix())
print(res, f"{res.stat().st_size / 1e6:.1f} MB")
print(cache, f"{cache.stat().st_size / 1e6:.1f} MB")